# Module 17: Remote Sensor Station

## Mission Briefing

Weather stations don't just read data -- they show it somewhere you can see. A thermometer on the roof isn't much use if you have to climb up to read it!

Today you combine **two sensors**, **WiFi**, and **a web page** into a real **remote sensor station**. You'll read temperature and light level, then serve a live dashboard you can view on your phone -- from anywhere in the room.

```
   Your Pico 2W                              Your Phone
   +--------------------+                    +------------------+
   |                    |                    |                  |
   | Thermistor  -> 24.3 C    ~~~ WiFi ~~~>   Temp: 24.3 C   |
   | Photoresistor -> 72%     ~~~ WiFi ~~~>   Light: 72%     |
   |                    |                    |                  |
   | (auto-refresh!)    |                    |  (updates every  |
   |                    |                    |   5 seconds!)    |
   +--------------------+                    +------------------+
```

No app to install. No button to press. Just open a browser and watch your sensor data update automatically!

---
## Science Spotlight: The Internet of Things

Real weather stations, smart thermostats, and factory monitors all do exactly what you're building today: read sensors and show the data on a screen somewhere else. This idea -- physical things connected to the internet -- is called the **Internet of Things (IoT)**.

There's a neat HTML trick that makes a web page reload itself automatically: the **auto-refresh meta tag**. No JavaScript needed!

*Your instructor will explain how IoT works, why auto-refresh is useful for sensor dashboards, and how real weather stations operate.*

---
## Components

You'll combine parts from Module 10 (photoresistor), Module 11 (thermistor), and Module 16 (web server).

| Component | Quantity | What It Does |
|-----------|----------|---------------|
| Pico 2W | 1 | Reads sensors + serves the web page |
| NTC Thermistor | 1 | Senses temperature (Module 11) |
| Photoresistor (LDR) | 1 | Senses light level (Module 10) |
| 10k ohm resistor | 2 | One for each voltage divider |
| Jumper wires | 6-7 | Connections |
| Breadboard | 1 | Base for your circuits |

---
## Wiring: Two Voltage Dividers

You're building TWO voltage divider circuits side by side -- one for temperature, one for light.

### Circuit Diagram

```
   Pico 2W
   +----------------------------------------------+
   |                                              |
   |   3.3V ──── Thermistor ──── junction A ─── GP26 (ADC0)
   |                                 |
   |                            [10k ohm]
   |                                 |
   |                                GND
   |                                              |
   |   3.3V ──── Photoresistor ──── junction B ─── GP27 (ADC1)
   |                                 |
   |                            [10k ohm]
   |                                 |
   |                                GND
   |                                              |
   +----------------------------------------------+
```

### Step-by-Step

**Temperature sensor (same as Module 11):**
1. Place the **thermistor** on the breadboard
2. Connect **3.3V** to one pin of the thermistor
3. Connect the other pin to **GP26** (this is junction A)
4. Place a **10k ohm resistor** from junction A to **GND**

**Light sensor (same as Module 10, but on GP27):**
5. Place the **photoresistor** on the breadboard (different rows!)
6. Connect **3.3V** to one pin of the photoresistor
7. Connect the other pin to **GP27** (this is junction B)
8. Place a **10k ohm resistor** from junction B to **GND**

Plug in your USB cable.

**Tip:** Keep the two voltage dividers on separate parts of the breadboard so the wires don't get tangled!

---
## Code Along Step 1: Test Both Sensors

Before we add WiFi, let's make sure both sensors are working.

Fill in the blanks:

In [ ]:
from machine import ADC
import math
import time

temp_adc = ADC(_____)              # Which GPIO pin is the thermistor on?
light_adc = ADC(_____)             # Which GPIO pin is the photoresistor on?

def read_temperature():
    """Read the thermistor and return temperature in Celsius."""
    val = temp_adc.read_u16()
    if val == 0:
        val = 1
    resistance = 10000 * val / (65535 - val)
    temp_k = 1 / (1/(273.15 + 25) + (1/3950) * math.log(resistance/10000))
    temp_c = temp_k - 273.15
    return temp_c

def read_light():
    """Read the photoresistor and return a percentage (0-100)."""
    val = light_adc.read_u16()
    percent = round(val / 65535 * 100)
    return percent

for i in range(10):
    temp = read_temperature()
    light = read_light()
    print("Temp:", round(temp, 1), "C  |  Light:", light, "%")
    time.sleep(1)

Try these quick tests:
- Does the temperature look reasonable (20-25 C)? ____________
- Cover the photoresistor -- does the light percentage drop? ____________
- Touch the thermistor -- does the temperature rise? ____________

If both sensors respond, you're ready for WiFi!

---
## Code Along Step 2: Connect to WiFi

Same WiFi setup from Module 15 and 16. Fill in your network name and password:

In [ ]:
import network
import time

SSID = "_____"                        # Your instructor will provide this
PASSWORD = "_____"                     # Your instructor will provide this

wlan = network.WLAN(network.STA_IF)
wlan.active(True)
wlan.connect(SSID, PASSWORD)

print("Connecting to WiFi...")
timeout = 10
while not wlan.isconnected() and timeout > 0:
    print("  Waiting...", timeout)
    time.sleep(1)
    timeout -= 1

if wlan.isconnected():
    ip = wlan.ifconfig()[0]
    print("Connected! IP address:", ip)
else:
    print("Failed to connect. Check SSID and password.")

Write down your IP address: ____________

---
## Code Along Step 3: Build the Sensor Dashboard Page

Now we build an HTML page that shows the sensor data. The magic ingredient is the **auto-refresh meta tag**:

```html
<meta http-equiv="refresh" content="5">
```

This tells the browser: "Reload this page every 5 seconds." Every time it reloads, the Pico reads the sensors again and sends fresh data!

Fill in the blanks:

In [ ]:
def build_page(temp_c, temp_f, light_pct):
    """Build an HTML dashboard showing sensor readings."""
    html = """<!DOCTYPE html>
<html>
<head>
    <title>Pico Sensor Station</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta http-equiv="_____" content="_____">   <!-- What attribute auto-refreshes? How many seconds? -->
    <style>
        body {
            font-family: Arial, sans-serif;
            text-align: center;
            background-color: #1a1a2e;
            color: white;
            margin: 0;
            padding: 20px;
        }
        h1 { font-size: 28px; margin-top: 30px; }
        .sensor-box {
            background-color: #16213e;
            border-radius: 15px;
            padding: 20px;
            margin: 15px auto;
            max-width: 300px;
        }
        .sensor-label { font-size: 18px; color: #aaa; }
        .sensor-value { font-size: 36px; font-weight: bold; margin: 10px 0; }
        .temp-color { color: #FF6B6B; }
        .light-color { color: #FFD93D; }
        .footer { margin-top: 30px; color: #555; font-size: 14px; }
    </style>
</head>
<body>
    <h1>Pico Sensor Station</h1>
    <div class="sensor-box">
        <div class="sensor-label">Temperature</div>
        <div class="sensor-value temp-color">""" + str(temp_c) + """ C</div>
        <div class="sensor-label">""" + str(temp_f) + """ F</div>
    </div>
    <div class="sensor-box">
        <div class="sensor-label">Light Level</div>
        <div class="sensor-value light-color">""" + str(light_pct) + """%</div>
    </div>
    <p class="footer">Auto-refreshes every 5 seconds</p>
    <p class="footer">Circuit Explorers - Module 17</p>
</body>
</html>
"""
    return html

# Test it!
print(build_page(23.5, 74.3, 65))

Look at the HTML output. Can you spot:
- The auto-refresh meta tag? ____________
- Where the temperature value appears? ____________
- Where the light percentage appears? ____________

---
## Code Along Step 4: The Full Sensor Station

Now we put everything together: WiFi + sensors + web server. This is the complete program!

Fill in the blanks:

In [ ]:
import network
import socket
import time
import math
from machine import ADC

# --- WiFi credentials ---
SSID = "_____"
PASSWORD = "_____"

# --- Set up sensors ---
temp_adc = ADC(_____)                  # Thermistor GPIO pin?
light_adc = ADC(_____)                 # Photoresistor GPIO pin?

def read_temperature():
    val = temp_adc.read_u16()
    if val == 0:
        val = 1
    resistance = 10000 * val / (65535 - val)
    temp_k = 1 / (1/(273.15 + 25) + (1/3950) * math.log(resistance/10000))
    temp_c = temp_k - 273.15
    return temp_c

def read_light():
    val = light_adc._____()            # What method reads the ADC?
    percent = round(val / 65535 * 100)
    return percent

def build_page(temp_c, temp_f, light_pct):
    html = """<!DOCTYPE html>
<html>
<head>
    <title>Pico Sensor Station</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta http-equiv="refresh" content="5">
    <style>
        body { font-family: Arial, sans-serif; text-align: center;
               background-color: #1a1a2e; color: white; padding: 20px; }
        h1 { font-size: 28px; margin-top: 30px; }
        .sensor-box { background-color: #16213e; border-radius: 15px;
                      padding: 20px; margin: 15px auto; max-width: 300px; }
        .sensor-label { font-size: 18px; color: #aaa; }
        .sensor-value { font-size: 36px; font-weight: bold; margin: 10px 0; }
        .temp-color { color: #FF6B6B; }
        .light-color { color: #FFD93D; }
        .footer { margin-top: 30px; color: #555; font-size: 14px; }
    </style>
</head>
<body>
    <h1>Pico Sensor Station</h1>
    <div class="sensor-box">
        <div class="sensor-label">Temperature</div>
        <div class="sensor-value temp-color">""" + str(temp_c) + """ C</div>
        <div class="sensor-label">""" + str(temp_f) + """ F</div>
    </div>
    <div class="sensor-box">
        <div class="sensor-label">Light Level</div>
        <div class="sensor-value light-color">""" + str(light_pct) + """%</div>
    </div>
    <p class="footer">Auto-refreshes every 5 seconds</p>
    <p class="footer">Circuit Explorers - Module 17</p>
</body>
</html>
"""
    return html

# --- Connect to WiFi ---
wlan = network.WLAN(network.STA_IF)
wlan.active(True)
wlan.connect(SSID, PASSWORD)

print("Connecting to WiFi...")
while not wlan.isconnected():
    time.sleep(1)

ip = wlan.ifconfig()[0]
print("Connected! IP:", ip)

# --- Create socket ---
addr = socket.getaddrinfo('0.0.0.0', 80)[0][-1]
s = socket.socket()
s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
s.bind(addr)
s.listen(_____)

print("\n** Sensor station running! **")
print("Open your browser to: http://" + ip)
print("Press Ctrl+C to stop\n")

# --- Main server loop ---
try:
    while True:
        cl, client_addr = s.accept()
        request = cl.recv(1024).decode()
        
        # Read sensors
        temp_c = round(read_temperature(), 1)
        temp_f = round(temp_c * _____ + _____, 1)    # Celsius to Fahrenheit?
        light_pct = read_light()
        
        print("Serving:", temp_c, "C /", temp_f, "F  Light:", light_pct, "%")
        
        # Send the web page
        response = build_page(temp_c, temp_f, light_pct)
        cl.send("HTTP/1.1 200 OK\r\n")
        cl.send("Content-Type: text/html\r\n")
        cl.send("Connection: close\r\n\r\n")
        cl.send(response)
        cl.close()

except KeyboardInterrupt:
    print("\nStation stopped.")
    s.close()

### Try it!

1. Run the code above
2. Wait for the IP address to appear
3. Open the IP address in your phone's browser
4. Watch the page -- it should refresh automatically every 5 seconds!

You just built an IoT device!

---
## Experiments

Try these while your sensor station is running:

1. **Phone test:** Open the dashboard on your phone and walk around the room. The page keeps refreshing as long as you're on the same WiFi!

2. **Cover the photoresistor:** Watch the light percentage drop on your phone's screen. Uncover it and watch it rise. You're monitoring a remote sensor!

3. **Warm the thermistor:** Hold the thermistor between your fingers. Watch the temperature climb on your phone. Let go and watch it cool down.

4. **Multiple viewers:** Ask a friend to open your IP address on their device. Can two people watch the same sensor station at the same time?

5. **Refresh speed:** Change the auto-refresh from 5 seconds to 2 seconds in the HTML. Does the data update faster?

---
## Challenge: Smart Alerts or History Log

Upgrade your sensor station with one of these features:

### Option A: Color-Coded Alerts
Change the temperature or light box color based on the reading:
- Temperature above 30 C? Make the box RED
- Light below 20%? Make the box DARK BLUE
- Everything normal? Keep the default colors

**Hint:** Use an `if` statement in `build_page()` to choose the color:
```python
if temp_c > 30:
    temp_bg = "#FF4444"    # Red alert!
else:
    temp_bg = "#16213e"    # Normal
```

### Option B: Reading History
Show the last 5 readings on the page, not just the current one.

**Hint:** Use a list to store recent readings:
```python
history = []
# In the server loop:
history.append(str(temp_c) + " C")
if len(history) > 5:
    history.pop(0)         # Remove the oldest
```

Then build an HTML list from the history in `build_page()`.

Ask your instructor for help if you get stuck!

In [ ]:
# Your challenge code here!


---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **IoT (Internet of Things)** | Physical devices connected to a network, sharing sensor data or receiving commands |
| **Auto-refresh** | An HTML meta tag that tells the browser to reload the page automatically on a timer |
| **Sensor station** | A device that reads one or more sensors and reports the data remotely |
| **HTTP response** | The data (HTML page) that a server sends back to a browser after a request |
| **Voltage divider** | Two resistors sharing voltage; used to read variable sensors like thermistors and photoresistors |
| **Dashboard** | A web page that displays live data in an easy-to-read format |

---
*Circuit Explorers -- Module 17: Remote Sensor Station*